# Build Pinecone Index — CLIP Fused Embeddings

Creates **5 namespaces** in one Pinecone index covering all ablation conditions:

| Namespace | Condition | CLIP weights | α |
|---|---|---|---|
| `frozen-alpha-1.0` | A | pretrained (frozen) | 1.0 |
| `frozen-alpha-0.7` | B | pretrained (frozen) | 0.7 |
| `frozen-alpha-0.5` | B | pretrained (frozen) | 0.5 |
| `finetuned-alpha-0.7` | C | clip_best.pt | 0.7 |
| `finetuned-alpha-0.5` | C | clip_best.pt | 0.5 |

In [1]:
!pip install -q pinecone open_clip_torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 38.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 15.6 MB/s eta 0:00:00


In [21]:
import os

# ══════════════════════════════════════════════════════════════════════════════
# CONFIG — fill in before running
# ══════════════════════════════════════════════════════════════════════════════
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY", "")
if not PINECONE_API_KEY:
    raise ValueError("Set PINECONE_API_KEY in the environment before running this notebook.")
INDEX_NAME = "vr-clothing-gallery"

DATASET_ROOT      = "/kaggle/input/datasets/vinay1706/deepfashion-cropped"   # update to your dataset path
CLIP_CHECKPOINT   = f"{DATASET_ROOT}/clip_seed_05.pt"
GALLERY_CSV       = f"{DATASET_ROOT}/gallery.csv"
CAPTIONS_CSV      = f"{DATASET_ROOT}/blip2_captions_gallery.csv"
IMAGE_ROOT        = DATASET_ROOT

CLIP_MODEL_NAME   = "ViT-L-14"
CLIP_PRETRAINED   = "openai"
EMBED_DIM         = 768
BATCH_SIZE        = 128
UPSERT_BATCH      = 100

NAMESPACES = [
    ("seed05-finetuned-alpha-0.7", True, 0.7),
    ("seed05-finetuned-alpha-0.5", True, 0.5),
]


In [22]:
import os
import torch
import torch.nn.functional as F
import open_clip
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from pinecone import Pinecone, ServerlessSpec
from itertools import groupby

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


In [23]:
# ── Load CSVs ─────────────────────────────────────────────────────────────────
gallery_df  = pd.read_csv(GALLERY_CSV)
captions_df = pd.read_csv(CAPTIONS_CSV)

df = gallery_df.merge(captions_df[["image_name", "blip2_caption"]], on="image_name", how="left")
df["blip2_caption"] = df["blip2_caption"].fillna("")

print(f"Gallery rows : {len(df)}")
print(f"Unique items : {df['item_id'].nunique()}")
df.head(3)

Gallery rows : 12612
Unique items : 3985


,image_name,item_id,evaluation_status,blip2_caption
0,img/img/WOMEN/Blouses_Shirts/id_00000001/02_1_...,id_00000001,gallery,"white blouse, black shorts"
1,img/img/WOMEN/Blouses_Shirts/id_00000001/02_3_...,id_00000001,gallery,"white blouse, black shorts"
2,img/img/WOMEN/Tees_Tanks/id_00000007/01_1_fron...,id_00000007,gallery,This is a tank top with a picture of snoop dog...


In [24]:
# ── Dataset ───────────────────────────────────────────────────────────────────
class GalleryDataset(Dataset):
    def __init__(self, df, image_root, preprocess):
        self.df         = df.reset_index(drop=True)
        self.image_root = image_root
        self.preprocess = preprocess

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_path = os.path.join(self.image_root, row["image_name"])
        try:
            image = self.preprocess(Image.open(img_path).convert("RGB"))
        except Exception:
            image = torch.zeros(3, 224, 224)
        return image, row["blip2_caption"], row["image_name"], str(row["item_id"])

In [25]:
# ── Helper: load CLIP (frozen or fine-tuned) ──────────────────────────────────
def load_clip(use_finetuned: bool):
    model, _, preprocess = open_clip.create_model_and_transforms(
        CLIP_MODEL_NAME, pretrained=CLIP_PRETRAINED
    )
    if use_finetuned:
        ckpt       = torch.load(CLIP_CHECKPOINT, map_location=device)
        state_dict = ckpt.get("model_state_dict", ckpt)
        state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}
        model.load_state_dict(state_dict, strict=False)
        print("  Loaded fine-tuned weights from clip_best.pt")
    else:
        print("  Using frozen pretrained weights")
    return model.to(device).eval(), preprocess, open_clip.get_tokenizer(CLIP_MODEL_NAME)

In [26]:
# ── Helper: embed entire gallery ──────────────────────────────────────────────
@torch.no_grad()
def embed_gallery(model, preprocess, tokenizer):
    dataset = GalleryDataset(df, IMAGE_ROOT, preprocess)
    loader  = DataLoader(dataset, batch_size=BATCH_SIZE,
                         num_workers=4, pin_memory=True, shuffle=False)
    vis_list, txt_list, names_list, ids_list = [], [], [], []

    with torch.amp.autocast('cuda'):
        for images, captions, image_names, item_ids in tqdm(loader, desc="  Embedding"):
            images = images.to(device)
            tokens = tokenizer(list(captions)).to(device)
            vis_emb = F.normalize(model.encode_image(images).float(), dim=-1)
            txt_emb = F.normalize(model.encode_text(tokens).float(),  dim=-1)
            vis_list.append(vis_emb.cpu())
            txt_list.append(txt_emb.cpu())
            names_list.extend(image_names)
            ids_list.extend(item_ids)

    return torch.cat(vis_list), torch.cat(txt_list), names_list, ids_list

In [27]:
# ── Helper: upsert one namespace ──────────────────────────────────────────────
def upsert_namespace(namespace, alpha, vis_embs, txt_embs, image_names, item_ids):
    fused = F.normalize(alpha * vis_embs + (1 - alpha) * txt_embs, dim=-1).numpy()
    vectors = [
        {
            "id"      : name.replace("/", "|"),
            "values"  : vec.tolist(),
            "metadata": {"item_id": item_id, "image_name": name, "alpha": alpha}
        }
        for vec, name, item_id in zip(fused, image_names, item_ids)
    ]
    for start in tqdm(range(0, len(vectors), UPSERT_BATCH),
                      desc=f"  Upserting {namespace}"):
        index.upsert(vectors=vectors[start:start + UPSERT_BATCH], namespace=namespace)
    print(f"  ✓ {len(vectors)} vectors → '{namespace}'")

In [28]:
import os

# ── Init Pinecone index ───────────────────────────────────────────────────────
pc = Pinecone(api_key=PINECONE_API_KEY)

if INDEX_NAME not in [idx.name for idx in pc.list_indexes()]:
    pc.create_index(
        name      = INDEX_NAME,
        dimension = EMBED_DIM,
        metric    = "cosine",
        spec      = ServerlessSpec(cloud="aws", region="us-east-1")
    )
    print(f"Index '{INDEX_NAME}' created.")
else:
    print(f"Index '{INDEX_NAME}' already exists — will add/overwrite namespaces.")

index = pc.Index(INDEX_NAME)
print(index.describe_index_stats())

Index 'vr-clothing-gallery' already exists — will add/overwrite namespaces.
DescribeIndexStatsResponse(dimension=768, total_vector_count=138732, metric='cosine', namespaces=11)


In [29]:
# ══════════════════════════════════════════════════════════════════════════════
# MAIN — embed once per model type, upsert all its namespaces
# ══════════════════════════════════════════════════════════════════════════════
for use_finetuned, group in groupby(NAMESPACES, key=lambda x: x[1]):
    group = list(group)
    label = "fine-tuned" if use_finetuned else "frozen pretrained"
    print(f"\n{'='*60}")
    print(f"Loading {label} CLIP...")

    model, preprocess, tokenizer = load_clip(use_finetuned)
    vis_embs, txt_embs, image_names, item_ids = embed_gallery(model, preprocess, tokenizer)
    print(f"  Embedded {len(image_names)} images.")

    for namespace, _, alpha in group:
        print(f"\nBuilding namespace '{namespace}' (α={alpha})...")
        upsert_namespace(namespace, alpha, vis_embs, txt_embs, image_names, item_ids)

    del model
    torch.cuda.empty_cache()

print("\n" + "="*60)
print("All namespaces built.")
print(index.describe_index_stats())


Loading fine-tuned CLIP...


/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


  Loaded fine-tuned weights from clip_best.pt


  Embedding:   0%|          | 0/99 [00:00<?, ?it/s]

  Embedded 12612 images.

Building namespace 'seed05-finetuned-alpha-0.7' (α=0.7)...


  Upserting seed05-finetuned-alpha-0.7:   0%|          | 0/127 [00:00<?, ?it/s]

  ✓ 12612 vectors → 'seed05-finetuned-alpha-0.7'

Building namespace 'seed05-finetuned-alpha-0.5' (α=0.5)...


  Upserting seed05-finetuned-alpha-0.5:   0%|          | 0/127 [00:00<?, ?it/s]

  ✓ 12612 vectors → 'seed05-finetuned-alpha-0.5'

All namespaces built.
DescribeIndexStatsResponse(dimension=768, total_vector_count=163956, metric='cosine', namespaces=13)


In [30]:
# ── Smoke test — top-3 from each namespace ────────────────────────────────────
# Use frozen model to generate a test query vector
test_model, test_prep, _ = load_clip(use_finetuned=False)
test_img_path = os.path.join(IMAGE_ROOT, df.iloc[0]["image_name"])
test_img = test_prep(Image.open(test_img_path).convert("RGB")).unsqueeze(0).to(device)
with torch.no_grad():
    test_vec = F.normalize(test_model.encode_image(test_img).float(), dim=-1)
test_vec = test_vec.cpu().numpy().tolist()[0]
del test_model
torch.cuda.empty_cache()

print("Smoke test — querying each namespace with gallery image 0:")
for namespace, _, alpha in NAMESPACES:
    result = index.query(vector=test_vec, top_k=3,
                         namespace=namespace, include_metadata=True)
    print(f"\n  [{namespace}]")
    for m in result["matches"]:
        print(f"    score={m['score']:.4f}  item={m['metadata']['item_id']}  img={m['metadata']['image_name']}")

  Using frozen pretrained weights
Smoke test — querying each namespace with gallery image 0:

  [seed05-finetuned-alpha-0.7]
    score=0.9358  item=id_00000001  img=img/img/WOMEN/Blouses_Shirts/id_00000001/02_1_front.jpg
    score=0.8771  item=id_00000001  img=img/img/WOMEN/Blouses_Shirts/id_00000001/02_3_back.jpg
    score=0.8735  item=id_00005874  img=img/img/WOMEN/Tees_Tanks/id_00005874/02_2_side.jpg

  [seed05-finetuned-alpha-0.5]
    score=0.7889  item=id_00000001  img=img/img/WOMEN/Blouses_Shirts/id_00000001/02_1_front.jpg
    score=0.7508  item=id_00004823  img=img/img/WOMEN/Blouses_Shirts/id_00004823/03_7_additional.jpg
    score=0.7468  item=id_00005353  img=img/img/WOMEN/Tees_Tanks/id_00005353/02_1_front.jpg
